In [17]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import h5py

import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix


import snntorch.functional

from ref_cnn_snn_model import CnnSnn

import snntorch.functional

import lava.lib.dl.slayer as slayer

from sklearn.model_selection import train_test_split

In [15]:
# =============================================================================
# Data loading — MODIFY THIS FOR YOUR DATASET
# =============================================================================
def load_data(
    df,
    train_ratio=0.6,
    valid_ratio=0.2,
    test_ratio=0.2,
    random_state=42,
    num_channels=14,
    window_size=384,     # e.g. 3 sec if fs=128
    stride=384,          # non-overlap; set 192 for 50% overlap
    drop_last=True,      # whether to drop the last incomplete window
):
    """
    Input df:
        one row = one channel of one full EEG trial

    Expected columns:
        - subject
        - video
        - channel
        - EEG_array   : full 1D EEG signal of that channel, shape (T_full,)
        - label

    Pipeline:
        1. segment each full trial signal into windows
        2. stack 14 channels into 2D EEG_array of shape (C, T_window)
        3. split by (subject, video), so all segments from one trial stay together

    Returns:
        train_dataset, valid_dataset, test_dataset, in_channels
    """

    # -----------------------------
    # 0) basic checks
    # -----------------------------
    required_cols = {"subject", "video", "channel", "EEG_clean", "label"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if not np.isclose(train_ratio + valid_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + valid_ratio + test_ratio must equal 1.0")

    df = df.copy()
    df = df.sort_values(["subject", "video", "channel"]).reset_index(drop=True)

    # -----------------------------
    # 1) segment full EEG trial first
    #    output rows: subject, video, channel, segment, EEG_segment, label
    # -----------------------------
    segmented_rows = []

    trial_group_cols = ["subject", "video", "channel"]
    grouped_trial_channel = df.groupby(trial_group_cols)

    for (sub, vid, ch), g in grouped_trial_channel:
        if len(g) != 1:
            raise ValueError(
                f"(subject={sub}, video={vid}, channel={ch}) has {len(g)} rows. "
                "Expected exactly 1 row per full-trial channel."
            )

        full_signal = np.asarray(g.iloc[0]["EEG_clean"], dtype=np.float32)
        label = int(g.iloc[0]["label"])

        if full_signal.ndim != 1:
            raise ValueError(
                f"(subject={sub}, video={vid}, channel={ch}) full EEG must be 1D, "
                f"got shape {full_signal.shape}"
            )

        T_full = len(full_signal)

        # if the total length is smaller then 1 window
        if T_full < window_size:
            # remove incomplete window, length alignment
            if drop_last:
                continue
            
            # add the padding to fill up the incomplete window
            else:
                padded = np.zeros(window_size, dtype=np.float32)
                padded[:T_full] = full_signal
                segmented_rows.append({
                    "subject": sub,
                    "video": vid,
                    "channel": ch,
                    "segment": 0,
                    "EEG_segment": padded,
                    "label": label,
                })
                continue

        seg_idx = 0
        for start in range(0, T_full, stride):
            end = start + window_size
            # Normal segment process
            if end <= T_full:
                seg = full_signal[start:end]

            # edge case
            else:
                # remove any incomplete window, length alignment
                if drop_last:
                    break
                seg = np.zeros(window_size, dtype=np.float32)
                valid_len = T_full - start
                if valid_len <= 0:
                    break
                seg[:valid_len] = full_signal[start:T_full]

            segmented_rows.append({
                "subject": sub,
                "video": vid,
                "channel": ch,
                "segment": seg_idx,
                "EEG_segment": seg,
                "label": label,
            })
            seg_idx += 1

    df_segch = pd.DataFrame(segmented_rows)

    if len(df_segch) == 0:
        raise ValueError("No segmented data generated. Check window_size/stride/drop_last.")

    # -----------------------------
    # 2) stack 14 channels -> 2D (C, T)
    #    one row becomes one (subject, video, segment)
    # -----------------------------
    df_segch = df_segch.sort_values(
        ["subject", "video", "segment", "channel"]
    ).reset_index(drop=True)

    group_cols = ["subject", "video", "segment"]
    grouped = df_segch.groupby(group_cols)

    rows = []
    for (sub, vid, seg), g in grouped:
        if len(g) != num_channels:
            raise ValueError(
                f"(subject={sub}, video={vid}, segment={seg}) "
                f"has {len(g)} channels, expected {num_channels}"
            )

        signals = []
        for _, row in g.iterrows():
            sig = np.asarray(row["EEG_segment"], dtype=np.float32)
            if sig.ndim != 1:
                raise ValueError(
                    f"(subject={sub}, video={vid}, segment={seg}) "
                    f"segment must be 1D, got shape {sig.shape}"
                )
            signals.append(sig)

        lengths = {len(s) for s in signals}
        if len(lengths) != 1:
            raise ValueError(
                f"(subject={sub}, video={vid}, segment={seg}) "
                f"channel segment lengths mismatch: {lengths}"
            )

        eeg_2d = np.stack(signals, axis=0)   # (C, T_window)
        label = int(g["label"].iloc[0])

        rows.append({
            "subject": sub,
            "video": vid,
            "segment": seg,
            "EEG_array": eeg_2d,
            "label": label,
        })

    df = pd.DataFrame(rows)

    # -----------------------------
    # 3) build group id = one full trial
    # -----------------------------
    df["group_id"] = df["subject"].astype(str) + "__" + df["video"].astype(str)
    unique_groups = df["group_id"].unique()

    # -----------------------------
    # 4) split by group, not by row
    # -----------------------------
    train_groups, temp_groups = train_test_split(
        unique_groups,
        test_size=(1.0 - train_ratio),
        random_state=random_state,
        shuffle=True,
    )

    valid_portion_of_temp = valid_ratio / (valid_ratio + test_ratio)

    valid_groups, test_groups = train_test_split(
        temp_groups,
        test_size=(1.0 - valid_portion_of_temp),
        random_state=random_state,
        shuffle=True,
    )

    train_df = df[df["group_id"].isin(train_groups)].copy()
    valid_df = df[df["group_id"].isin(valid_groups)].copy()
    test_df  = df[df["group_id"].isin(test_groups)].copy()

    # -----------------------------
    # 5) Convert table -> tensors
    # -----------------------------
    def df_to_dataset(split_df):
        if len(split_df) == 0:
            raise ValueError("One split is empty. Adjust split ratios or dataset size.")

        x_list = []
        y_list = []

        for _, row in split_df.iterrows():
            x = np.asarray(row["EEG_array"], dtype=np.float32)   # (C, T_window)

            if x.ndim != 2:
                raise ValueError(
                    f"Each EEG_array must have shape (C, T), got shape {x.shape}"
                )

            x_list.append(x)
            y_list.append(int(row["label"]))

        x = np.stack(x_list, axis=0)   # (N, C, T)
        y = np.asarray(y_list, dtype=np.int64)

        x_tensor = torch.from_numpy(x).float()
        y_tensor = torch.from_numpy(y).long()
        return TensorDataset(x_tensor, y_tensor)

    train_dataset = df_to_dataset(train_df)
    valid_dataset = df_to_dataset(valid_df)
    test_dataset  = df_to_dataset(test_df)

    # -----------------------------
    # 6) infer input channels
    # -----------------------------
    sample_x = np.asarray(df.iloc[0]["EEG_array"], dtype=np.float32)
    in_channels = sample_x.shape[0]

    # -----------------------------
    # 7) logging
    # -----------------------------
    print("=== Group-wise split summary ===")
    print(f"Total segment-samples: {len(df)}")
    print(f"Total trials(groups): {len(unique_groups)}")
    print(f"Train segments: {len(train_df)} | groups: {len(train_groups)}")
    print(f"Valid segments: {len(valid_df)} | groups: {len(valid_groups)}")
    print(f"Test  segments: {len(test_df)} | groups: {len(test_groups)}")
    print(f"in_channels: {in_channels}")
    print(f"window_size: {window_size}, stride: {stride}")
    print("No trial is split across train/valid/test.")

    return train_dataset, valid_dataset, test_dataset

In [12]:
def mat_dataset_load(path = '/Users/linyuchun/Desktop/Project/SNN/data/EEG_clean_table.csv', eeg_prefix = 'EEG_clean'):
    seg_eeg = pd.read_csv(path)
    eeg_cols = [c for c in seg_eeg.columns if c.startswith(eeg_prefix)]

    seg_eeg[eeg_prefix] = (
        seg_eeg[eeg_cols]
        .apply(lambda row: row.dropna().astype(float).values, axis=1)
    )

    seg_eeg = seg_eeg.drop(columns=eeg_cols)
    return seg_eeg